# Step 1: Install Dependencies

In [16]:
# !pip install langchain openai faiss-cpu tiktoken
# !pip install -U langchain-community
# !pip install python-dotenv
# !pip install unstructured
# !pip install "unstructured[pdf]"
# !pip install rouge-score

# Step 2: Load and Embed Documents

In [ ]:
# Document Loading & Splitting
from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Embedding & Vector Storage
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS

# Environment Variables & API Setup
from dotenv import load_dotenv
import openai
import os

# Retrieval-Based QA Pipeline
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

# ROUGE Scoring for Evaluation
from rouge_score import rouge_scorer


# Step 3: Loading OpenAI API Key from .env File

In [18]:
# Set OpenAI API Key
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")
# Check if key is loaded (for debugging)
if openai.api_key:
    print("API Key loaded successfully!")
else:
    print("Failed to load API Key. Check your .env file.")


API Key loaded successfully!


# Step 4: Loading, Splitting, and Storing Documents in FAISS for Retrieval

In [19]:
# Load documents from a folder
loader = DirectoryLoader("documents/", glob="*.pdf")  # Adjust for PDFs, CSVs, etc.
documents = loader.load()

# Split documents into chunks for better retrieval
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

# Convert text chunks into embeddings and store in FAISS
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)

# Save the FAISS index for later use
vectorstore.save_local("faiss_index")

In [20]:
# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
# Load FAISS index
vectorstore = FAISS.load_local("faiss_index", allow_dangerous_deserialization=True, embeddings=embeddings)

# Create a retriever
retriever = vectorstore.as_retriever()

# Use OpenAI's LLM
llm = ChatOpenAI(model_name="gpt-4", temperature=0)

# Build the RAG pipeline
qa_chain = RetrievalQA.from_chain_type(llm, retriever=retriever)
results = []

absolute_path = "./questions/Questions.csv"
questions_df = pd.read_csv(absolute_path)

for question_no, row in questions_df.iterrows():
    question = row["Question"]
    human_answer = row["Answer"] 

    prompt = f"""
        Provide a response to the following question in no more than 20 words.

        {question}

        Answer: """
    
    response = qa_chain.run(prompt)
    
    # Compute ROUGE scores
    rouge_scores = scorer.score(human_answer, response)

    # Print results
    print(f"Question {question_no + 1}: {question}")
    print(f"Human Answer: {human_answer}")
    print(f"Response: {response}")
    print("")
    print(f"ROUGE-1: {rouge_scores['rouge1'].fmeasure}")
    print(f"ROUGE-2: {rouge_scores['rouge2'].fmeasure}")
    print(f"ROUGE-L: {rouge_scores['rougeL'].fmeasure}")
    print("")
    print("------------------------------------------------")
    
    # Store results as a dictionary
    results.append({
        "question": question,
        "human_answer": human_answer,
        "response": response,
        "rouge1": rouge_scores["rouge1"].fmeasure,
        "rouge2": rouge_scores["rouge2"].fmeasure,
        "rougeL": rouge_scores["rougeL"].fmeasure,
    })

df = pd.DataFrame(results)
df.to_csv("./questions/results.csv", index=False)


C:\Users\User\AppData\Local\Temp\ipykernel_7912\2710579100.py:30: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = qa_chain.run(prompt)


Question 1: What are the primary compounds in coffee?
Human Answer: Caffeine, chlorogenic acid, cafestol, kahweol, carbohydrates, lipids, nitrogenous compounds, vitamins, and minerals.
Response: Coffee primarily contains caffeine, carbohydrates, lipids, nitrogenous compounds, vitamins, minerals, alkaloids, and phenolic compounds.

ROUGE-1: 0.6153846153846153
ROUGE-2: 0.33333333333333337
ROUGE-L: 0.5384615384615384

------------------------------------------------
Question 2: What is the origin of coffee consumption?
Human Answer: Coffee consumption began in northeast Africa and spread to the Middle East in the 15th century.
Response: Coffee consumption probably originated in northeast Africa and spread to the Middle East in the 15th century.

ROUGE-1: 0.9090909090909091
ROUGE-2: 0.8387096774193549
ROUGE-L: 0.9090909090909091

------------------------------------------------
Question 3: How does caffeine affect the nervous system?
Human Answer: Coffee blocks adenosine receptors, reducin

# Step 7: Computing Overall ROUGE Scores for Model Evaluation

In [22]:
overall_rouge = {
    "ROUGE-1": df["rouge1"].mean(),
    "ROUGE-2": df["rouge2"].mean(),
    "ROUGE-L": df["rougeL"].mean(),
}

# Print overall ROUGE scores
print("Overall ROUGE Scores:")
for key, value in overall_rouge.items():
    print(f"{key}: {value:.4f}")

Overall ROUGE Scores:
ROUGE-1: 0.5193
ROUGE-2: 0.2772
ROUGE-L: 0.4444
